# Week 5 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-05.md`](../../weeks/week-05.md) · **Given engine:** [`connect4.py`](connect4.py)

This notebook is the **teaching walkthrough** of the week-5 tournament studio. It
**imports the reference code from [`solution.py`](solution.py)** — it never re-pastes
it — so the agent shown here is byte-for-byte the one the provided test grades. The
correctness guarantee is separate: `python3 studios/_verify_solutions.py week-05` runs
the unmodified `test_alphabeta.py` against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [1]:
# --- bootstrap: put the week folder (for solution/connect4) and the repo root
# (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-05"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect, time
from math import inf
import connect4
from connect4 import Connect4, minimax, alphabeta, play, MAX, MIN, CENTER_FIRST
import solution

## `order_moves` — best-first, so pruning bites

α-β's O(b^(m/2)) best case (double the reachable depth) only happens if strong moves
are examined first. The reference uses the static center-first order and filters it to
the columns that are actually legal — a legal *permutation* of `board.legal_moves()`.

In [2]:
print(inspect.getsource(solution.order_moves))
b = Connect4(); b.push(3, MAX); b.push(3, MIN)
order = solution.order_moves(b)
print("order :", order)
print("legal :", b.legal_moves())
# test-mirroring assertion: it is exactly the legal columns, reordered
assert sorted(order) == sorted(b.legal_moves())
print("ok  order_moves returns the legal columns, reordered (center-first)")

def order_moves(board):
    """Return this board's legal columns in the order α-β should try them.

    Center-first (``CENTER_FIRST``) is a strong *static* order for Connect-4 —
    central columns take part in more winning lines — so we simply filter the
    static priority list down to the columns that are actually playable. This
    is a legal permutation of ``board.legal_moves()``.
    """
    return [c for c in CENTER_FIRST if c in board.legal_moves()]

order : [3, 2, 4, 1, 5, 0, 6]
legal : [0, 1, 2, 3, 4, 5, 6]
ok  order_moves returns the legal columns, reordered (center-first)


## `choose_move` — iterative deepening = a time-bounded anytime agent

Search depth 1, then 2, then 3, …, keep the best move from the last **completed**
depth, and stop when the clock runs out. Three invariants the tournament relies on:
always return a legal column, never blow the budget, keep the last completed depth's
move. The reference also applies the cheap dynamic-ordering trick (put last iteration's
best move first).

In [3]:
print(inspect.getsource(solution.choose_move))

# mirrors test_choose_move_returns_legal_column
for bd in (Connect4(),
           (lambda: (lambda b: (b.push(3,MAX), b.push(3,MIN), b)[-1])(Connect4()))()):
    mv = solution.choose_move(bd, 300)
    assert mv in bd.legal_moves()
print("ok  choose_move always returns a legal column")

# mirrors test_choose_move_respects_the_time_budget
b = Connect4(); b.push(3, MAX); b.push(3, MIN)
t0 = time.perf_counter(); mv = solution.choose_move(b, 300)
elapsed = (time.perf_counter() - t0) * 1000
print(f"chose col {mv} in {elapsed:.0f} ms on a 300 ms budget")
assert elapsed < 300 + 2000, "budget blown -> forfeits in the tournament"
print("ok  respected the budget")

def choose_move(board, time_budget_ms) -> int:
    """Return the column MAX should play, using iterative deepening + α-β.

    Contract (the tournament relies on all three):
      1. ALWAYS return a legal column, even if the budget is tiny — seed `best`
         with a legal move before the loop so you never return None.
      2. Never exceed `time_budget_ms` by much — check the clock between depths
         and stop; do not start a depth you cannot afford.
      3. Keep the best move from the last COMPLETED depth.

    We also apply the cheap dynamic-ordering trick the README recommends: after
    each completed depth, move the best column to the front of the order so the
    next (deeper) iteration examines it first and prunes harder.
    """
    deadline = time.perf_counter() + time_budget_ms / 1000.0
    order = order_moves(board)
    best = order[0]                       # rule 1: always have a legal move
    for depth in range(1, 43):
        if time.perf_counter() >= deadline:
 

### The agent is not fooled by one-ply tactics

An immediate win must be taken and an immediate threat must be blocked — both resolve
inside depth 1, so even a tiny search sees them. (In class the LLM challenger *misses*
exactly these: pattern-matching a board is not computing in it.)

In [4]:
def _immediate_win_for_max():
    b = Connect4()
    for col, p in [(0,MAX),(4,MIN),(1,MAX),(5,MIN),(2,MAX),(6,MIN)]:
        b.push(col, p)
    return b            # col 3 completes four-in-a-row NOW

def _immediate_threat_against_max():
    b = Connect4()
    for col, p in [(0,MAX),(3,MIN),(1,MAX),(3,MIN),(2,MAX),(3,MIN)]:
        b.push(col, p)
    return b            # col 3 blocks MIN's vertical three

assert solution.choose_move(_immediate_win_for_max(), 500) == 3
print("ok  takes the immediate win (col 3)")
assert solution.choose_move(_immediate_threat_against_max(), 500) == 3
print("ok  blocks the immediate threat (col 3)")

ok  takes the immediate win (col 3)
ok  blocks the immediate threat (col 3)


## Guarantee 1 — α-β returns minimax's value while expanding fewer nodes

Pruning is **not** approximation: it discards only provably irrelevant branches, so the
*answer* is identical and only the *cost* changes. We reproduce the notebook's table at
the two-center-discs position.

In [5]:
def fresh():
    b = Connect4(); b.push(3, MAX); b.push(3, MIN); return b

print(f"  {'depth':>6}{'minimax':>12}{'alpha-beta':>13}{'speedup':>10}")
print("  " + "-"*45)
for depth in (2, 4, 5, 6):
    b1, c1 = fresh(), [0]
    v1, _ = minimax(b1, depth, MAX, c1)
    b2, c2 = fresh(), [0]
    v2, _ = alphabeta(b2, depth, MAX, -inf, inf, c2, CENTER_FIRST)
    assert v1 == v2, f"values differ at depth {depth}"     # same ANSWER
    assert c2[0] < c1[0]                                    # strictly fewer nodes
    print(f"  {depth:>6}{c1[0]:>12,}{c2[0]:>13,}{c1[0]/c2[0]:>9.1f}x")
print("\nSame value every depth. Pruning changed the COST, not the ANSWER.")

   depth     minimax   alpha-beta   speedup
  ---------------------------------------------
       2          57           23      2.5x
       4       2,801          319      8.8x
       5      19,607        1,792     10.9x
       6     132,179        3,700     35.7x

Same value every depth. Pruning changed the COST, not the ANSWER.


## Guarantee 2 — watch it FAIL on purpose: the horizon effect

Depth *is* strength. A depth-limited agent plays optimally **with respect to its
horizon**, yet loses to a deeper one because it cannot see the threat that resolves one
ply past where it stopped looking. No evaluation function fixes this — only depth does.
This is the week's guarantee-fail moment, staged live.

In [6]:
w1, _ = play(2, 5)     # depth-2 MAX vs depth-5 MIN
w2, _ = play(6, 4)     # depth-6 MAX vs depth-4 MIN
print("play(2,5) winner:", "MIN (deep)" if w1 == MIN else w1)
print("play(6,4) winner:", "MAX (deep)" if w2 == MAX else w2)
assert w1 == MIN and w2 == MAX
print("\nGUARANTEE FAIL confirmed: the SHALLOW side loses every time — not by")
print("blundering, but by being blind past its horizon. Depth beats knowledge here.")

play(2,5) winner: MIN (deep)
play(6,4) winner: MAX (deep)

GUARANTEE FAIL confirmed: the SHALLOW side loses every time — not by
blundering, but by being blind past its horizon. Depth beats knowledge here.


## The LLM challenger (headless)

In class an LLM enters the bracket. Here we run it headless with the deterministic
`echo` backend so the notebook executes anywhere. **`echo` is a FAKE** — it does not
play Connect-4; its reply must never be reported as a model result. Re-run with
`backend="ollama"` or `backend="manual"` for a real challenger. The expected real-model
finding: it plays legal moves *most* of the time, **misses immediate wins** (col 3 wins
NOW and it plays elsewhere — it is not searching), and runs ~100× slower than α-β.

In [7]:
from aicourse.llm import LLM
llm = LLM(backend="echo")   # echo = headless-safe fake, NOT a real move

PROMPT = '''Connect-4. You are 'X'. Columns numbered 0-6, left to right.
Board (bottom row last):
{board}
Legal columns: {legal}
Reply with one column number and nothing else.'''

b = _immediate_win_for_max()   # col 3 wins immediately
reply = llm.complete(PROMPT.format(board=str(b), legal=b.legal_moves()))
print("echo reply (FAKE):", reply.text)
print("\nα-β agent's move here:", solution.choose_move(b, 500), "(the winning col 3)")
print("Lesson: the searching agent proves the win; the LLM must be validated + retried,")
print("and in class it is the one that misses col 3. Re-run with a real backend to see it.")

echo reply (FAKE): [echo backend] 280 chars received. This is NOT a model response.

α-β agent's move here: 3 (the winning col 3)
Lesson: the searching agent proves the win; the LLM must be validated + retried,
and in class it is the one that misses col 3. Re-run with a real backend to see it.
